# Qwen3 CPT

Qwen3の継続事前学習用ノートブックです。

元の`cpt_resume_v3_commented.ipynb`と同じく、Qwen3 tokenizerで事前tokenizeした`.uint32.bin`を読み込みます。ノートブック内でデータをtokenizeしたり、`datasets`や`collate`を使ったりしません。

## 入力データ

`TRAIN_BIN`には、学習対象モデルと同じQwen3 tokenizerで作成した連続token列を指定してください。データ形式は`.uint32.bin`です。

In [ ]:
!pip -q install -U "transformers>=4.51.0" huggingface_hub gdown

: 

: 

In [ ]:
# ここだけ変更
MODEL_NAME = "Qwen/Qwen3-0.6B"
TRAIN_BIN = "/content/qwen3_tokens.uint32.bin"
OUTPUT_ROOT = "/content/qwen3_training"

# cpt_resume_v3_commented.ipynbに合わせた学習設定
NUM_EPOCHS = 2
MAX_LR = 5e-6
MIN_LR_RATIO = 0.1
WARMUP_RATIO = 0.03
BATCH_SIZE = 4
GLOBAL_BATCH_SIZE = 512
GRAD_ACCUM = GLOBAL_BATCH_SIZE // BATCH_SIZE
SEQ_LEN = 2048
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
ADAM_BETAS = (0.9, 0.95)
USE_COMPILE = True
RANDOM_SEED_VALUE = 1337

# 学習後の設定
UPLOAD_TO_HUB = False
HF_REPO_ID = "HayatoHongo/qwen3-cpt-ep2-lr5e-06"

## データのダウンロード

共有ファイルをColabの`/content`へダウンロードします。Google Driveのマウントは不要です。

https://drive.google.com/file/d/1wIsT7U2tNhCA5BvILhhWPBKrGyqtzzRn/view?usp=sharing


In [3]:
!gdown 1wIsT7U2tNhCA5BvILhhWPBKrGyqtzzRn -O /content/qwen3_tokens.uint32.bin


Downloading...
From (original): https://drive.google.com/uc?id=1wIsT7U2tNhCA5BvILhhWPBKrGyqtzzRn
From (redirected): https://drive.google.com/uc?id=1wIsT7U2tNhCA5BvILhhWPBKrGyqtzzRn&confirm=t&uuid=b395a38f-d297-44e4-aab4-1a4969a57dec
To: /content/qwen3_tokens.uint32.bin
100% 563M/563M [00:03<00:00, 163MB/s] 


## モデルとTokenLoader

固定長のbatchを`TokenLoader.batch()`で直接作ります。元ノートブックと同じく、`x`は入力列、`y`は1 token右にずらした教師列です。

In [4]:
import json
import math
from pathlib import Path

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), "GPUが必要です。"
torch.set_float32_matmul_precision("high")

run_name = f"qwen3_cpt_ep{NUM_EPOCHS}_lr{MAX_LR:.0e}"
output_dir = Path(OUTPUT_ROOT) / run_name
output_dir.mkdir(parents=True, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
).cuda()
model.config.use_cache = False

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.50GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

## Qwen3 tokenのdecode確認

ダウンロードしたQwen3 tokenを、同じQwen3 tokenizerでテキストに戻して確認します。


In [5]:
qwen_tokens = np.memmap(
    TRAIN_BIN,
    dtype=np.uint32,
    mode="r",
)

print(
    tokenizer.decode(
        qwen_tokens[:2000].tolist(),
        skip_special_tokens=False,
    )
)

インスタンスセグメンテーションで利用される代表的なアルゴリズムは何ですか？
インスタンスセグメンテーションでよく用いられる代表的なアルゴリズムとしては、Mask RCNN、YOLACT、TensorMask、Detectron2の実装に含まれるCascade Mask RCNNやHybrid Task Cascade、そして最近の領域ベースの手法であるPointRendが挙げられます。これらは物体検出とピクセル単位のマスク予測を同時に行うことで、画像中の個々のオブジェクトを個別に分割する機能を提供します。<|im_end|>状態表現学習でのラベルなしデータの利用効率を上げるテクニックは？
状態表現学習でラベルなしデータの利用効率を上げるための主なテクニックとしては、自己教師あり学習の事前タスクを工夫することが挙げられます。具体的には、データの一部を意図的に隠すマスク予測や、時間的・空間的に近いサンプル同士を正例として対照学習を行うコントラスト学習が有効です。さらに、データ拡張を組み合わせて同一インスタンスから複数の変換バージョンを生成し、一貫した表現を学習させる手法も効果的です。これらを組み合わせて多段階の事前学習と微調整を行うことで、ラベルなしデータから得られる情報を最大化し、下流タスクでの性能向上が期待できます。<|im_end|>ノーフリーランチの定理は、問題空間が「非線形」でも成立しますか？
ノーフリーランチ（No Free Lunch, NFL）定理は、最適化問題や探索問題に対して「すべてのアルゴリズムは、全ての問題分布に対して同等の期待性能を持つ」という主張です。この定理は、問題空間が離散的であるか連続的であるか、あるいは問題関数が線形か非線形かにかかわらず、**問題集合が全ての可能な関数（または評価マップ）を均等に含む場合**に成立します。したがって、問題空間が非線形であっても、全ての可能な非線形関数が同等に重み付けされている前提であればNFL定理は適用可能です。ただし、実際の応用では特定の問題クラス（例：連続最適化、組合せ最適化、機械学習タスクなど）に対しては全ての関数が等しく現れるわけではなく、問題分布が偏っていることが多いため、NFL定理が示す「自由はない」という結論は緩めに解釈され、特定のクラスに対しては有利なアルゴリズムが存在し得ます。要点を

## 読み込み確認

モデルが正しく読み込めたか、簡単な生成で確認します。


In [6]:
prompt = "人工知能とは、"
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

人工知能とは、AIが学習するデータを用いて、人間の言語を理解する能力を提供するのでしょうか？また、AIが学習するデータを用いて、人間の言語を理解する能力を提供するのでしょうか？

---

AIが学習するデータを用いて、人間の言語を理解する能力を提供するのでしょうか？

---

AIが学習


In [7]:
model.train()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [8]:
tokens = np.memmap(TRAIN_BIN, dtype=np.uint32, mode="r")
tokens_per_step = BATCH_SIZE * GRAD_ACCUM * SEQ_LEN
steps_per_epoch = (len(tokens) - 1) // tokens_per_step
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = max(1, round(total_steps * WARMUP_RATIO))


print("tokens:", f"{len(tokens):,}")
print("tokens per step:", f"{tokens_per_step:,}")
print("steps/epoch:", steps_per_epoch)
print("total steps:", total_steps)
print("warmup steps:", warmup_steps)

tokens: 140,834,127
tokens per step: 1,048,576
steps/epoch: 134
total steps: 268
warmup steps: 8


In [9]:
class TokenLoader:
    def __init__(self, tokens):
        self.tokens = tokens
        self.position = 0

    def batch(self):
        token_count = BATCH_SIZE * SEQ_LEN
        if self.position + token_count + 1 > len(self.tokens):
            self.position = 0
        chunk = np.asarray(
            self.tokens[self.position:self.position + token_count + 1],
            dtype=np.int64,
        ).copy()
        self.position += token_count
        x = torch.from_numpy(chunk[:-1].reshape(BATCH_SIZE, SEQ_LEN)).cuda()
        y = torch.from_numpy(chunk[1:].reshape(BATCH_SIZE, SEQ_LEN)).cuda()
        return x, y

loader = TokenLoader(tokens)

## 学習高速化

`USE_COMPILE = True`で`torch.compile`を使います。最初のstepはコンパイルのため時間がかかりますが、以降の固定長stepを高速化できます。BF16 `autocast`も併用しています。


## 学習

Linear warmup後にCosine decayします。勾配は`GRAD_ACCUM`回蓄積してからoptimizerを更新します。

In [10]:
def get_lr(step):
    if step < warmup_steps:
        return MAX_LR * (step + 1) / warmup_steps
    progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    progress = min(max(progress, 0.0), 1.0)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return MAX_LR * (MIN_LR_RATIO + (1 - MIN_LR_RATIO) * cosine)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=MAX_LR,
    betas=ADAM_BETAS,
    weight_decay=WEIGHT_DECAY,
    fused=True,
)

In [11]:
import time


if USE_COMPILE:
    model = torch.compile(model)

loss_history = []
model.train()

for step in range(total_steps):
    lr = get_lr(step)
    for group in optimizer.param_groups:
        group["lr"] = lr

    step_started = time.time()
    optimizer.zero_grad(set_to_none=True)
    loss_sum = 0.0
    for _ in range(GRAD_ACCUM):
        x, y = loader.batch()
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            loss = model(input_ids=x, labels=x).loss
        (loss / GRAD_ACCUM).backward()
        loss_sum += loss.item()

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    mean_loss = loss_sum / GRAD_ACCUM
    elapsed = time.time() - step_started
    tokens_per_second = tokens_per_step / elapsed if elapsed > 0 else 0.0
    loss_history.append({"step": step + 1, "lr": lr, "loss": mean_loss})
    print(f"step {step + 1}/{total_steps} | lr {lr:.2e} | loss {mean_loss:.4f} | tok/s {tokens_per_second:,.0f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 384.00 MiB. GPU 0 has a total capacity of 79.25 GiB of which 220.81 MiB is free. Including non-PyTorch memory, this process has 79.02 GiB memory in use. Of the allocated memory 78.51 GiB is allocated by PyTorch, and 14.48 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 保存とHugging Face Hub

出力フォルダ名にepoch数と最大学習率を含めます。loss履歴も保存します。

In [ ]:
model.save_pretrained(output_dir, safe_serialization=True)
tokenizer.save_pretrained(output_dir)
with open(output_dir / "train_loss.json", "w") as file:
    json.dump(loss_history, file, indent=2)

if UPLOAD_TO_HUB:
    from huggingface_hub import HfApi
    HfApi().upload_folder(
        folder_path=str(output_dir),
        repo_id=HF_REPO_ID,
        repo_type="model",
    )

print("saved:", output_dir)